# Bullet Fusion Model

The decision and action classifiers (`03_decision_classifier.ipynb`, `04_action_extractor.ipynb`) flag
**individual transcript sentences**. Showing those raw sentences as bullet points looks disjointed —
consecutive flagged sentences are often fragments of the same decision or action, spoken across several
turns, with disfluencies and no shared subject:

```text
B: you've got yourself and the Industrial Designer gonna be working quite closely at this stage
B: looking from prototyping of some des description using clay
B: Um looks like we're gonna need a maybe a kind of prototype in clay
```

This notebook trains a small **fusion/compression model** that rewrites a group of noisy evidence
sentences into one clean, third-person bullet — the same style AMI's human annotators used:

```text
The industrial designer and user interface designer will work on a prototype.
```

**Where the training pairs come from.** AMI's manual annotations link every human-written abstractive
sentence (in `<decisions>` / `<actions>` inside `*.abssumm.xml`) back to the raw dialogue-act utterances
that justify it, via `*.summlink.xml`. Each dialogue act points to a word range inside
`<meeting>.<speaker>.words.xml`. Chaining these three files gives free, human-quality
(noisy evidence -> clean bullet) supervision, no manual labeling needed:

```text
summlink.xml -> dialog-act.xml (word range) -> words.xml (raw text)   [noisy evidence]
summlink.xml -> abssumm.xml (sentence text)                          [clean target]
```

**Model.** `t5-small` fine-tuned with a task prefix, so one model handles both bullet types:

- `"fuse decision: <evidence>"` -> clean decision bullet
- `"fuse action: <evidence>"` -> clean action bullet

**Runtime use (`06_meeting_demo.ipynb`).** Adjacent transcript chunks flagged by the same classifier are
grouped into an evidence group, fed through this model, and the single generated bullet is displayed
instead of the raw flagged sentences. For action items, only the action *phrase* is rewritten — the
assignee and deadline extracted by the BIO tagger are kept as-is and shown alongside, so the model can't
hallucinate a person or date that never appeared in the transcript.

## 1. Setup

In [ ]:
import inspect
import random
import re
import subprocess
import sys
import warnings
import xml.etree.ElementTree as ET
import zipfile
from collections import defaultdict
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "transformers>=4.40", "datasets", "evaluate", "rouge-score",
         "sentencepiece", "accelerate"],
        check=True,
    )

import evaluate
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
assert DEVICE == "cuda", "Enable a T4 GPU: Runtime -> Change runtime type -> T4 GPU."
print(f"Device: {torch.cuda.get_device_name(0)}")

def find_root():
    for candidate in [
        Path("."), Path(".."), Path("/content"), Path("/content/project"),
        Path("/content/drive/MyDrive/Graduation Project"),
    ]:
        if (candidate / "amicorpus" / "abstractive").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find amicorpus/abstractive. Set ROOT to the folder containing amicorpus/."
    )

ROOT = find_root()
AMI_ROOT = ROOT / "amicorpus"
MODELS_DIR = ROOT / "models"
OUTPUT_DIR = MODELS_DIR / "bullet_fusion_t5"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# The AMI manual-annotations archive. Only needed once, to unzip extractive/
# and dialogueActs/ into amicorpus/. Point this at wherever you uploaded it.
MANUAL_ANNOTATIONS_ZIP = ROOT / "ami_public_manual_1.6.2.zip"

MODEL_NAME = "t5-small"
MAX_SOURCE_TOKENS = 160
MAX_TARGET_TOKENS = 64

print("Project:", ROOT)
print("Output :", OUTPUT_DIR)

## 2. Unzip the `extractive/` and `dialogueActs/` annotations

`01_data_preparation.ipynb` already unzips `words/` and `abstractive/` from
`ami_public_manual_1.6.2.zip`. This notebook additionally needs `extractive/` (the
`*.summlink.xml` alignment files) and `dialogueActs/` (the `*.dialog-act.xml` word-range files).
This cell extracts only those two folders, and skips extraction entirely if they already exist.

In [ ]:
EXTRACTIVE_DIR = AMI_ROOT / "extractive"
DIALOGUE_ACTS_DIR = AMI_ROOT / "dialogueActs"

needed = [d for d in [EXTRACTIVE_DIR, DIALOGUE_ACTS_DIR] if not d.is_dir() or not any(d.iterdir())]
if needed:
    if not MANUAL_ANNOTATIONS_ZIP.exists():
        raise FileNotFoundError(
            f"Need to extract {[d.name for d in needed]} but {MANUAL_ANNOTATIONS_ZIP} is missing.\n"
            "Upload ami_public_manual_1.6.2.zip to the project root, or update MANUAL_ANNOTATIONS_ZIP."
        )
    with zipfile.ZipFile(MANUAL_ANNOTATIONS_ZIP) as zf:
        members = [
            name for name in zf.namelist()
            if name.startswith("extractive/") or name.startswith("dialogueActs/")
        ]
        zf.extractall(AMI_ROOT, members=members)
    print(f"Extracted {len(members)} files into {AMI_ROOT}")
else:
    print("extractive/ and dialogueActs/ already present, skipping extraction.")

print("extractive files    :", len(list(EXTRACTIVE_DIR.glob("*.xml"))))
print("dialogueActs files  :", len(list(DIALOGUE_ACTS_DIR.glob("*.xml"))))

## 3. Chain summlink -> dialog-act -> words into (evidence, target, task_type) pairs

- `parse_abssumm_sections` reads `<decisions>` / `<actions>` sentences (with their `nite:id`) from
  `*.abssumm.xml` — these are the **clean targets**.
- `parse_summlinks` reads `*.summlink.xml` and groups the extractive dialogue-act references that
  justify each abstractive sentence.
- `WordIndex` + `resolve_dact_href` + `get_text_for_range` turn a dialogue-act's word range into
  detokenized **raw evidence text**.

In [ ]:
NITE_NS = "{http://nite.sourceforge.net/}"
HREF_RE = re.compile(r"([^#]+)#id\(([^)]+)\)(?:\.\.id\(([^)]+)\))?")


def parse_abssumm_sections(path):
    """Return {sentence_id: (text, section)} for the decisions/actions sections of one meeting."""
    root = ET.parse(path).getroot()
    out = {}
    for section in ("decisions", "actions"):
        node = root.find(section)
        if node is None:
            continue
        for sent in node.findall("sentence"):
            sentence_id = sent.attrib.get(f"{NITE_NS}id")
            text = (sent.text or "").strip()
            if sentence_id and text:
                out[sentence_id] = (text, section)
    return out


def parse_summlinks(path):
    """Return {abstractive_sentence_id: set of extractive dialogue-act hrefs}."""
    root = ET.parse(path).getroot()
    groups = defaultdict(set)
    for link in root.findall("summlink"):
        extractive_href = abstractive_href = None
        for pointer in link.findall(f"{NITE_NS}pointer"):
            role, href = pointer.attrib.get("role"), pointer.attrib.get("href")
            if role == "extractive":
                extractive_href = href
            elif role == "abstractive":
                abstractive_href = href
        if extractive_href and abstractive_href:
            match = HREF_RE.match(abstractive_href)
            if match:
                groups[match.group(2)].add(extractive_href)
    return groups

In [ ]:
class WordIndex:
    """Lazy nite:id -> (text, is_word, starttime) lookup for one <meeting>.<speaker>.words.xml file."""

    _cache = {}

    @classmethod
    def get(cls, meeting, speaker):
        key = (meeting, speaker)
        if key not in cls._cache:
            path = AMI_ROOT / "words" / f"{meeting}.{speaker}.words.xml"
            ordered = []
            if path.exists():
                root = ET.parse(path).getroot()
                for element in root:
                    tag = element.tag.rsplit("}", 1)[-1]
                    element_id = element.attrib.get(f"{NITE_NS}id")
                    if element_id is None:
                        continue
                    start = float(element.attrib.get("starttime", "0") or 0.0)
                    if tag == "w":
                        text = (element.text or "").strip()
                        is_word = element.attrib.get("punc") != "true" and bool(text)
                        ordered.append((element_id, text, is_word, start))
                    else:
                        ordered.append((element_id, "", False, start))  # vocalsound, disfmarker, gap
            cls._cache[key] = ordered
        return cls._cache[key]


def resolve_dact_href(dact_href, dialog_act_cache):
    """'<meeting>.<speaker>.dialog-act.xml#id(...)' -> (meeting, speaker, start_word_id, end_word_id)."""
    match = HREF_RE.match(dact_href)
    if not match:
        return None
    filename, dact_id, _ = match.groups()
    meeting, speaker = filename.replace(".dialog-act.xml", "").rsplit(".", 1)

    cache_key = (meeting, speaker)
    if cache_key not in dialog_act_cache:
        path = DIALOGUE_ACTS_DIR / filename
        lookup = {}
        if path.exists():
            root = ET.parse(path).getroot()
            for dact in root.findall("dact"):
                dact_element_id = dact.attrib.get(f"{NITE_NS}id")
                child = dact.find(f"{NITE_NS}child")
                if dact_element_id and child is not None:
                    lookup[dact_element_id] = child.attrib.get("href")
        dialog_act_cache[cache_key] = lookup

    word_href = dialog_act_cache[cache_key].get(dact_id)
    if not word_href:
        return None
    word_match = HREF_RE.match(word_href)
    if not word_match:
        return None
    _, start_id, end_id = word_match.groups()
    return meeting, speaker, start_id, end_id or start_id


def get_text_for_range(meeting, speaker, start_id, end_id):
    """Detokenize the words between two nite:id word-range endpoints, plus the span's start time."""
    ordered = WordIndex.get(meeting, speaker)
    ids_in_order = [row[0] for row in ordered]
    try:
        start_pos, end_pos = ids_in_order.index(start_id), ids_in_order.index(end_id)
    except ValueError:
        return None, None
    if end_pos < start_pos:
        start_pos, end_pos = end_pos, start_pos

    words, start_time = [], None
    for _, text, is_word, timestamp in ordered[start_pos:end_pos + 1]:
        if start_time is None:
            start_time = timestamp
        if is_word:
            words.append(text)
    if not words:
        return None, start_time
    joined = " ".join(words)
    for punct in [",", ".", "?", "!", ";", ":"]:
        joined = joined.replace(f" {punct}", punct)
    return joined, start_time

In [ ]:
def build_pairs_for_meeting(meeting_id):
    summary_path = AMI_ROOT / "abstractive" / f"{meeting_id}.abssumm.xml"
    summlink_path = EXTRACTIVE_DIR / f"{meeting_id}.summlink.xml"
    if not summary_path.exists() or not summlink_path.exists():
        return []

    target_sentences = parse_abssumm_sections(summary_path)
    groups = parse_summlinks(summlink_path)
    dialog_act_cache = {}
    pairs = []

    for sentence_id, dact_hrefs in groups.items():
        if sentence_id not in target_sentences:
            continue  # link targets an <abstract> sentence, not a decision/action
        target_text, section = target_sentences[sentence_id]

        resolved = []
        for href in dact_hrefs:
            ref = resolve_dact_href(href, dialog_act_cache)
            if ref is None:
                continue
            meeting, speaker, start_id, end_id = ref
            text, start_time = get_text_for_range(meeting, speaker, start_id, end_id)
            if text:
                resolved.append((start_time or 0.0, speaker, text))
        if not resolved:
            continue

        resolved.sort(key=lambda row: row[0])  # chronological reading order
        deduped = []
        for _, speaker, text in resolved:
            if not deduped or deduped[-1] != (speaker, text):
                deduped.append((speaker, text))

        pairs.append({
            "meeting_id": meeting_id,
            "sentence_id": sentence_id,
            "task_type": "decision" if section == "decisions" else "action",
            "n_evidence": len(deduped),
            "source": " ".join(f"{speaker}: {text}" for speaker, text in deduped),
            "target": target_text,
        })
    return pairs


print("Parsing functions ready.")

## 4. Build the full pair dataset

Every meeting with `extractive/*.summlink.xml` data contributes pairs. This yields roughly 880 pairs
across 135 meetings (~624 decisions, ~256 actions) — small, but each pair is a clean human-quality
target, and `t5-small` needs far less data than a summarizer trained from scratch.

In [ ]:
meeting_ids = sorted({path.stem.split(".")[0] for path in EXTRACTIVE_DIR.glob("*.summlink.xml")})
print(f"Meetings with summlink data: {len(meeting_ids)}")

all_pairs = []
for meeting_id in meeting_ids:
    all_pairs.extend(build_pairs_for_meeting(meeting_id))

pairs_df = pd.DataFrame(all_pairs)
print(f"Total pairs: {len(pairs_df)}")
print(pairs_df["task_type"].value_counts())
print(f"\nMeetings contributing at least one pair: {pairs_df['meeting_id'].nunique()}")

source_lengths = pairs_df["source"].str.split().str.len()
target_lengths = pairs_df["target"].str.split().str.len()
print(f"\nSource words -> mean {source_lengths.mean():.1f}, p95 {source_lengths.quantile(0.95):.0f}, max {source_lengths.max()}")
print(f"Target words -> mean {target_lengths.mean():.1f}, p95 {target_lengths.quantile(0.95):.0f}, max {target_lengths.max()}")
print(f"Evidence utterances per bullet -> mean {pairs_df['n_evidence'].mean():.2f}, max {pairs_df['n_evidence'].max()}")

In [ ]:
print("=== Sample pairs ===\n")
for row in pairs_df.sample(n=min(6, len(pairs_df)), random_state=SEED).itertuples():
    print(f"[{row.task_type}] ({row.n_evidence} evidence utterances)")
    print(f"  SOURCE: {row.source[:220]}")
    print(f"  TARGET: {row.target}")
    print()

## 5. Meeting-level train/validation/test split

Splitting by `meeting_id` (not by pair) prevents leakage — the same meeting's vocabulary and
phrasing never appears in both train and test.

In [ ]:
shuffled_meetings = meeting_ids.copy()
random.Random(SEED).shuffle(shuffled_meetings)

n = len(shuffled_meetings)
n_test = max(1, round(n * 0.15))
n_val = max(1, round(n * 0.15))
test_meetings = set(shuffled_meetings[:n_test])
val_meetings = set(shuffled_meetings[n_test:n_test + n_val])
train_meetings = set(shuffled_meetings[n_test + n_val:])

def split_of(meeting_id):
    if meeting_id in test_meetings:
        return "test"
    if meeting_id in val_meetings:
        return "validation"
    return "train"

pairs_df["split"] = pairs_df["meeting_id"].map(split_of)
print(pairs_df.groupby(["split", "task_type"]).size().unstack(fill_value=0))
print(f"\nMeetings -> train {len(train_meetings)}, validation {len(val_meetings)}, test {len(test_meetings)}")

## 6. Tokenize with task prefixes

T5 is prompted for the task via a text prefix on the input, e.g. `"fuse decision: <evidence>"`.
This lets one shared model learn both bullet styles instead of training two separate models.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def add_prefix(row):
    return f"fuse {row['task_type']}: {row['source']}"

pairs_df["prefixed_source"] = pairs_df.apply(add_prefix, axis=1)

def to_dataset(split_name):
    subset = pairs_df[pairs_df["split"] == split_name]
    return Dataset.from_pandas(subset[["prefixed_source", "target", "task_type", "meeting_id"]], preserve_index=False)

raw_datasets = DatasetDict({
    "train": to_dataset("train"),
    "validation": to_dataset("validation"),
    "test": to_dataset("test"),
})

def tokenize_pairs(batch):
    model_inputs = tokenizer(batch["prefixed_source"], max_length=MAX_SOURCE_TOKENS, truncation=True)
    labels = tokenizer(text_target=batch["target"], max_length=MAX_TARGET_TOKENS, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized = raw_datasets.map(tokenize_pairs, batched=True, remove_columns=raw_datasets["train"].column_names)
print(tokenized)

## 7. Fine-tune `t5-small`

`t5-small` (~60M params) trains in a few minutes on a T4 for a dataset this size. Early stopping on
validation ROUGE-L keeps the model from overfitting the ~600-pair training set.

In [ ]:
rouge = evaluate.load("rouge")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)

def trainer_metrics(eval_pred):
    predictions, labels = eval_pred
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    predictions = np.asarray(predictions)
    if predictions.ndim == 3:
        predictions = predictions.argmax(axis=-1)
    # Seq2SeqTrainer pads with -100; tokenizers reject negative ids, so remap before decoding.
    predictions = np.where(predictions < 0, tokenizer.pad_token_id, predictions).astype(np.int64)
    labels = np.where(labels < 0, tokenizer.pad_token_id, labels).astype(np.int64)
    decoded_predictions = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = rouge.compute(predictions=decoded_predictions, references=decoded_labels, use_stemmer=True)
    return {"rouge1": result["rouge1"], "rouge2": result["rouge2"], "rougeL": result["rougeL"]}

training_kwargs = dict(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    num_train_epochs=25,
    learning_rate=3e-4,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,
    per_device_eval_batch_size=16,
    fp16=True,
    weight_decay=0.01,
    warmup_ratio=0.10,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_TOKENS,
    generation_num_beams=4,
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    greater_is_better=True,
    logging_steps=10,
    report_to="none",
    seed=SEED,
)
evaluation_key = "eval_strategy" if "eval_strategy" in inspect.signature(Seq2SeqTrainingArguments).parameters else "evaluation_strategy"
training_kwargs[evaluation_key] = "epoch"
training_args = Seq2SeqTrainingArguments(**training_kwargs)

trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100),
    compute_metrics=trainer_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)],
)
# Newer Transformers renamed the tokenizer= argument to processing_class=.
tokenizer_key = "processing_class" if "processing_class" in inspect.signature(Seq2SeqTrainer).parameters else "tokenizer"
trainer_kwargs[tokenizer_key] = tokenizer

trainer = Seq2SeqTrainer(**trainer_kwargs)
trainer.train()

FUSION_MODEL_PATH = OUTPUT_DIR / "bullet_fusion_t5"
trainer.save_model(FUSION_MODEL_PATH)
tokenizer.save_pretrained(FUSION_MODEL_PATH)
print("Saved ->", FUSION_MODEL_PATH)

## 8. Evaluate on held-out test meetings

ROUGE against the human-written AMI bullet, computed overall and separately for decisions vs. actions.

In [ ]:
@torch.no_grad()
def generate_bullets(prefixed_sources, batch_size=16):
    model.eval()
    outputs = []
    for start in range(0, len(prefixed_sources), batch_size):
        batch = prefixed_sources[start:start + batch_size]
        encoded = tokenizer(batch, truncation=True, padding=True,
                            max_length=MAX_SOURCE_TOKENS, return_tensors="pt").to(DEVICE)
        generated = model.generate(**encoded, num_beams=4, max_new_tokens=MAX_TARGET_TOKENS)
        outputs.extend(tokenizer.batch_decode(generated, skip_special_tokens=True))
    return outputs

def rouge_scores(predictions, references):
    scores = rouge.compute(predictions=predictions, references=references, use_stemmer=True)
    return {
        "ROUGE-1": round(scores["rouge1"] * 100, 2),
        "ROUGE-2": round(scores["rouge2"] * 100, 2),
        "ROUGE-L": round(scores["rougeL"] * 100, 2),
    }

test_df = raw_datasets["test"].to_pandas()
test_df["generated"] = generate_bullets(test_df["prefixed_source"].tolist())

print("Overall:", rouge_scores(test_df["generated"].tolist(), test_df["target"].tolist()))
for task_type, group in test_df.groupby("task_type"):
    print(f"{task_type:>9}:", rouge_scores(group["generated"].tolist(), group["target"].tolist()),
          f"(n={len(group)})")

## 9. Qualitative comparison

Side-by-side view of the noisy evidence, the gold AMI bullet, and the model's generated bullet —
this is what actually matters for the demo, more than the ROUGE number above.

In [ ]:
sample = test_df.sample(n=min(8, len(test_df)), random_state=SEED)
for row in sample.itertuples():
    evidence = row.prefixed_source.split(": ", 1)[1]
    print(f"[{row.task_type}]")
    print(f"  EVIDENCE  : {evidence[:220]}")
    print(f"  GOLD      : {row.target}")
    print(f"  GENERATED : {row.generated}")
    print()

## 10. Inference helper

`fuse_bullets` is the function `06_meeting_demo.ipynb` calls: give it a list of raw flagged
sentences (as `"SPEAKER: text"` strings, or plain text) plus `task_type`, get back one clean bullet.

In [ ]:
@torch.no_grad()
def fuse_bullets(evidence_sentences, task_type="decision"):
    """Rewrite a list of noisy flagged sentences into one clean bullet.

    evidence_sentences: list[str], each optionally already formatted as "SPEAKER: text".
    task_type: "decision" or "action".
    """
    if not evidence_sentences:
        return ""
    source = " ".join(evidence_sentences)
    prefixed = f"fuse {task_type}: {source}"
    encoded = tokenizer(prefixed, truncation=True, max_length=MAX_SOURCE_TOKENS, return_tensors="pt").to(DEVICE)
    generated = model.generate(**encoded, num_beams=4, max_new_tokens=MAX_TARGET_TOKENS)
    return tokenizer.decode(generated[0], skip_special_tokens=True)


# Sanity check with the disconnected-sentence example that motivated this notebook.
demo_evidence = [
    "B: you've got yourself and the Industrial Designer gonna be working quite closely at this stage",
    "B: looking from prototyping of some des description using clay",
    "B: Um looks like we're gonna need a maybe a kind of prototype in clay",
]
print("Raw flagged sentences (what the classifiers alone produce):")
for line in demo_evidence:
    print(" ", line)
print("\nFused bullet:")
print(" ", fuse_bullets(demo_evidence, task_type="action"))